In [3]:
import pandas as pd  # Import pandas for structured data analysis.
import numpy as np  # Import NumPy for numerical diagnostics.
from pathlib import Path  # Import Path for file-safe references.
from IPython.display import display  # Import display for readable audit tables.
from scipy import stats  # Import statistical tests for audit checks.
FILE_PATH = "raw_data/returns.csv"  # Point to the project raw dataset.
audit_findings = []  # Create a register for evidence-based findings.
print("BUSINESS INSIGHT: Customer Retention and Churn")  # State the consulting context for the audit.
print("BUSINESS PROBLEM: Identify customer behaviours associated with inactivity and churn.")  # State the business problem being investigated.
print("AUDIT LENS: retention, repeat purchase, payment friction, service experience")  # State the signals relevant to this problem.


BUSINESS INSIGHT: Customer Retention and Churn
BUSINESS PROBLEM: Identify customer behaviours associated with inactivity and churn.
AUDIT LENS: retention, repeat purchase, payment friction, service experience


In [4]:
path_candidates = [
    Path(FILE_PATH),
    Path.cwd() / FILE_PATH,
    Path.cwd().parent / FILE_PATH,
    Path.cwd().parent.parent / FILE_PATH,
    Path.cwd().parent.parent.parent / FILE_PATH,
]

data_path = next((path for path in path_candidates if path.is_file()), None)

if data_path is None:
    raise FileNotFoundError(
        f"Could not locate {FILE_PATH!r}. Checked: "
        + ", ".join(str(path) for path in path_candidates)
    )

df = pd.read_csv(data_path)

print(f"Loaded {data_path.name}")
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns):,}")
display(df.head())  # Load the raw dataset before making changes.
print(f"Loaded {Path(FILE_PATH).name}")  # Confirm the source dataset loaded.
print(f"Rows: {len(df):,}")  # Show the available observation count.
print(f"Columns: {len(df.columns):,}")  # Show the available field count.
display(df.head())  # Inspect representative raw records.


Loaded returns.csv
Rows: 5,938
Columns: 10


,return_id,order_item_id,customer_id,return_request_date,return_received_date,return_reason,return_condition,refund_amount,refund_status,return_channel
0,RET-00000001,ITM-000000018,CUS-006161,2022-06-06,2022-06-19,Size/Fit,Resellable,61.65,Processed,Store
1,RET-00000002,ITM-000000042,CUS-008027,2024-09-03,2024-09-13,Changed Mind,Damaged,171.42,Approved,Courier
2,RET-00000003,ITM-000000051,CUS-004390,2020-11-15,2020-11-25,Changed Mind,Opened,99.35,Pending,Store
3,RET-00000004,ITM-000000081,CUS-006336,2020-07-07,2020-07-12,NaN,Opened,322.62,Approved,Courier
4,RET-00000005,ITM-000000104,CUS-000691,2022-06-22,2022-06-22,Defective Product,Damaged,115.69,Approved,Courier


Loaded returns.csv
Rows: 5,938
Columns: 10


,return_id,order_item_id,customer_id,return_request_date,return_received_date,return_reason,return_condition,refund_amount,refund_status,return_channel
0,RET-00000001,ITM-000000018,CUS-006161,2022-06-06,2022-06-19,Size/Fit,Resellable,61.65,Processed,Store
1,RET-00000002,ITM-000000042,CUS-008027,2024-09-03,2024-09-13,Changed Mind,Damaged,171.42,Approved,Courier
2,RET-00000003,ITM-000000051,CUS-004390,2020-11-15,2020-11-25,Changed Mind,Opened,99.35,Pending,Store
3,RET-00000004,ITM-000000081,CUS-006336,2020-07-07,2020-07-12,NaN,Opened,322.62,Approved,Courier
4,RET-00000005,ITM-000000104,CUS-000691,2022-06-22,2022-06-22,Defective Product,Damaged,115.69,Approved,Courier


In [5]:
overview = pd.DataFrame({"metric":["rows","columns","duplicates","missing_cells"],"value":[len(df),len(df.columns),int(df.duplicated().sum()),int(df.isna().sum().sum())]})  # Build an initial data-quality summary.
display(overview)  # Review the initial quality position.
print("Decision point: determine which findings require remediation.")  # Make the audit decision explicit.


,metric,value
0,rows,5938
1,columns,10
2,duplicates,0
3,missing_cells,689


Decision point: determine which findings require remediation.


In [6]:
schema = pd.DataFrame({"column":df.columns,"dtype":df.dtypes.astype(str).values,"non_null":df.notna().sum().values,"missing":df.isna().sum().values,"unique":df.nunique(dropna=True).values})  # Profile schema completeness and cardinality.
display(schema)  # Inspect field-level structural evidence.
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()  # Identify numeric fields for statistical checks.
text_cols = df.select_dtypes(include=["object","string"]).columns.tolist()  # Identify text fields for categorical checks.
date_cols = [c for c in df.columns if "date" in c.lower() or "time" in c.lower()]  # Identify likely temporal fields.


,column,dtype,non_null,missing,unique
0,return_id,str,5938,0,5938
1,order_item_id,str,5938,0,5938
2,customer_id,str,5724,214,4374
3,return_request_date,str,5938,0,2229
4,return_received_date,str,5938,0,2213
5,return_reason,str,5463,475,7
6,return_condition,str,5938,0,4
7,refund_amount,float64,5938,0,3170
8,refund_status,str,5938,0,3
9,return_channel,str,5938,0,2


In [7]:
missing = df.isna().sum().sort_values(ascending=False)  # Measure explicit missingness by field.
missing = missing[missing.gt(0)]  # Keep only fields with missing values.
display(missing.to_frame("missing_count"))  # Inspect missing-value concentration.
for col in missing.index: audit_findings.append({"issue":"missing_values","column":col,"count":int(missing[col])})  # Register observed missing-value evidence.


,missing_count
return_reason,475
customer_id,214


In [8]:
duplicates = int(df.duplicated().sum())  # Measure exact duplicate records.
audit_findings.append({"issue":"duplicate_rows","count":duplicates})  # Register duplicate-row evidence.
print(f"Duplicate rows identified: {duplicates:,}")  # Report duplicate-row evidence.


Duplicate rows identified: 0


In [9]:
numeric_audit = df[numeric_cols].describe().T if numeric_cols else pd.DataFrame()  # Summarise numeric distributions.
display(numeric_audit)  # Inspect scale, spread and potential extremes.
if numeric_cols: outlier_rates = ((df[numeric_cols] < df[numeric_cols].quantile(.25) - 1.5*(df[numeric_cols].quantile(.75)-df[numeric_cols].quantile(.25))) | (df[numeric_cols] > df[numeric_cols].quantile(.75) + 1.5*(df[numeric_cols].quantile(.75)-df[numeric_cols].quantile(.25)))).mean().sort_values(ascending=False)  # Estimate IQR-based extreme-value rates.
if numeric_cols: display(outlier_rates.to_frame("iqr_extreme_rate"))  # Inspect fields requiring business review.


,count,mean,std,min,25%,50%,75%,max
refund_amount,5938.0,105.851568,118.052953,3.4,39.7425,70.02,123.27,1613.73


,iqr_extreme_rate
refund_amount,0.077636


In [10]:
category_audit = []  # Create categorical consistency checks.
for col in text_cols: category_audit.append({"column":col,"unique":int(df[col].nunique(dropna=True)),"blank":int(df[col].astype("string").str.strip().eq("").sum()),"top_values":df[col].value_counts(dropna=False).head(5).to_dict()})  # Profile text fields for inconsistent values.
display(pd.DataFrame(category_audit))  # Review categorical concentration and blanks.


,column,unique,blank,top_values
0,return_id,5938,0,"{'RET-00000001': 1, 'RET-00000002': 1, 'RET-00..."
1,order_item_id,5938,0,"{'ITM-000000018': 1, 'ITM-000000042': 1, 'ITM-..."
2,customer_id,4374,0,"{nan: 214, 'CUS-001525': 5, 'CUS-000160': 5, '..."
3,return_request_date,2229,0,"{'2023-12-22': 11, '2026-04-09': 10, '2022-10-..."
4,return_received_date,2213,0,"{'2024-05-16': 9, '2025-08-19': 9, '2022-02-09..."
5,return_reason,7,0,"{'Changed Mind': 1678, 'Defective Product': 10..."
6,return_condition,4,0,"{'Damaged': 1534, 'Resellable': 1500, 'Defecti..."
7,refund_status,3,0,"{'Pending': 2007, 'Approved': 1988, 'Processed..."
8,return_channel,2,0,"{'Courier': 3403, 'Store': 2535}"


In [11]:
date_audit = []  # Create temporal field diagnostics.
for col in date_cols: parsed = pd.to_datetime(df[col], errors="coerce"); date_audit.append({"column":col,"parse_failures":int(parsed.isna().sum()-df[col].isna().sum()),"min":parsed.min(),"max":parsed.max()})  # Test temporal fields for parseability and range.
display(pd.DataFrame(date_audit))  # Review date integrity before analysis.


,column,parse_failures,min,max
0,return_request_date,0,2020-01-06,2026-10-13
1,return_received_date,0,2020-01-18,2026-10-17


In [12]:
identifier_audit = []  # Create identifier uniqueness diagnostics.
for col in df.columns: identifier_audit.append({"column":col,"unique_ratio":round(df[col].nunique(dropna=True)/max(len(df),1),3)})  # Measure field-level uniqueness.
identifier_audit = pd.DataFrame(identifier_audit).sort_values("unique_ratio",ascending=False)  # Rank potential identifiers and keys.
display(identifier_audit.head(15))  # Inspect candidate identifiers and high-cardinality fields.


,column,unique_ratio
0,return_id,1.000
1,order_item_id,1.000
2,customer_id,0.737
7,refund_amount,0.534
3,return_request_date,0.375
4,return_received_date,0.373
5,return_reason,0.001
6,return_condition,0.001
8,refund_status,0.001
9,return_channel,0.000


In [13]:
numeric_pairs = []  # Create relationship diagnostics for numeric fields.
if len(numeric_cols) > 1: numeric_pairs = df[numeric_cols].corr(numeric_only=True).stack().reset_index(name="correlation")  # Measure numeric relationships for diagnostic context.
if numeric_pairs != []: display(numeric_pairs.sort_values("correlation",key=lambda s:s.abs(),ascending=False).head(20))  # Inspect strongest observed numeric relationships.


In [14]:
finding_table = pd.DataFrame(audit_findings)  # Convert findings into a reviewable audit register.
if finding_table.empty: finding_table = pd.DataFrame([{ "issue":"none_detected_by_template", "count":0 }])  # Record when automated checks find no issues.
display(finding_table)  # Review the evidence before remediation.
print("Consulting decision: validate material findings against business rules before cleaning.")  # Prevent automatic treatment of every anomaly as an error.


,issue,column,count
0,missing_values,return_reason,475
1,missing_values,customer_id,214
2,duplicate_rows,NaN,0


Consulting decision: validate material findings against business rules before cleaning.


In [15]:
stem = Path(FILE_PATH).stem  # Capture the dataset stem for output naming.
audit_summary = pd.DataFrame({"dataset":[Path(FILE_PATH).name],"rows":[len(df)],"columns":[len(df.columns)],"duplicates":[duplicates],"missing_cells":[int(df.isna().sum().sum())]})  # Create an auditable executive summary.
display(audit_summary)  # Present the final audit snapshot.
audit_summary.to_csv(f"outputs/{stem}_audit_summary.csv",index=False)  # Save the audit summary for downstream review.


,dataset,rows,columns,duplicates,missing_cells
0,returns.csv,5938,10,0,689
